# 01 - EEG source reconstruction: inverse methods and regularization

**Source reconstruction hands-on** — dataset `ds000117`, subject `sub-01`,
face-recognition paradigm, **EEG channels only**.

Everything that comes before the inverse problem (preprocessing, epoching, ICA,
averaging, noise covariance, source space, BEM, coregistration, forward solution)
has **already been computed** and lives in the `data/` folder. Here we focus on a
single question:

> how does the reconstructed cortical activity change when we change the
> **inverse method** and the **regularization parameter**?

## What we will do in this notebook

1. load the prepared data (3 evoked responses, noise covariance, forward solution);
2. look at the three evoked responses — `famous`, `unfamiliar`, `scrambled`;
3. build the inverse operator;
4. compute a reference solution and display it on the cortical surface;
5. extract time courses in a few anatomical **ROIs**;
6. run the whole **method × regularization** grid and save the results.

The comparison figures are made in `02_visualization.ipynb`.

## 0. Setup

The only thing you may need to change are the paths, if you moved the folder.

In [ ]:
%matplotlib inline

from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne
from mne.minimum_norm import make_inverse_operator, apply_inverse, write_inverse_operator

mne.set_log_level("warning")

# This notebook lives in notebooks/: go up to the project root.
PROJECT = Path.cwd()
if not (PROJECT / "data").exists() and (PROJECT.parent / "data").exists():
    PROJECT = PROJECT.parent

DATA = PROJECT / "data"
SUBJECTS_DIR = PROJECT / "subjects" #contiene l'anatomia FreeSurfer
RESULTS = PROJECT / "results" #conterrà ciò che calcoliamo durante la hands-on
STC_DIR = RESULTS / "stc" #contiene forward, Evoked, covariance ecc.
FIG_DIR = RESULTS / "figures"
for d in (RESULTS, STC_DIR, FIG_DIR):  #Creazione automatica delle cartelle di output
    d.mkdir(parents=True, exist_ok=True)

SUBJECT = "sub-01"
CONDITIONS = ["famous", "unfamiliar", "scrambled"] #definizione delle condizioni
COLORS = {"famous": "#1f77b4", "unfamiliar": "#ff7f0e", "scrambled": "#2ca02c"}

print("project :", PROJECT)
print("data    :", DATA)
print("anatomy :", SUBJECTS_DIR)

In [ ]:
print(mne.__version__)

In [ ]:
# 3D backend.
#  - "pyvistaqt": opens a separate interactive window (recommended)
#  - "notebook" : draws the brain inside the notebook, but needs the trame
#                 packages (trame, trame-vtk, trame-vuetify)
mne.viz.set_3d_backend("pyvistaqt")
print("3D backend:", mne.viz.get_3d_backend())
print("MNE-Python:", mne.__version__)

#Qui diciamo a MNE: quando visualizzi il cervello 3D, usa PyVistaQt.

## 1. Loading the prepared data

| file | content |
|---|---|
| `sub-01-eeg-ave.fif` | the three evoked responses, already averaged and kept separate |
| `sub-01-eeg-cov.fif` | noise covariance, estimated on the pre-stimulus baseline |
| `sub-01-eeg-fwd.fif` | EEG forward solution (leadfield), 3-layer BEM |
| `sub-01-src.fif` | source space on the cortical surface |
| `sub-01-trans.fif` | head ↔ MRI transformation (coregistration) |

All the parameters used to generate them are stored in `data/provenance.json`.

In [ ]:
with open(DATA / "provenance.json", encoding="utf-8") as fid:
    provenance = json.load(fid)

# provenance.json -> file che documenta come sono stati preparati
print(json.dumps(provenance["prepared_here"], indent=2, ensure_ascii=False))

In [ ]:
# proj=True applies the average-reference projector (mandatory for EEG modelling)

#carico i 3 evoked (famous, unfamiliar, scrambled) e li metto in un dizionario
evokeds = {
    cond: mne.read_evokeds(DATA / f"{SUBJECT}-eeg-ave.fif", condition=cond, proj=True)
    for cond in CONDITIONS
}

noise_cov = mne.read_cov(DATA / f"{SUBJECT}-eeg-cov.fif") #caricamento matrice di covarianza del rumore
fwd = mne.read_forward_solution(DATA / f"{SUBJECT}-eeg-fwd.fif") #Caricamento del forward model

#Per ciascuna condizione stampa: numero trial mediati, numero canali EEG, intervallo temporale
for cond, ev in evokeds.items():
    print(f"{cond:<12} {ev.nave:>4} averaged epochs, "
          f"{len(ev.ch_names)} EEG channels, "
          f"{ev.times[0]:+.3f} .. {ev.times[-1]:+.3f} s")

n_sens, n_dip = fwd["sol"]["data"].shape #dimensione della matrice leadfield: n_sens, n_dip
print(f"\nleadfield: {n_sens} sensors x {n_dip} dipoles ({fwd['nsource']} sources)")

## 2. The three evoked responses

The three conditions are kept **separate**: famous faces, unfamiliar faces and
scrambled faces (low-level control). We do not compute any contrast: each condition
will be reconstructed on its own.

In [ ]:
# In this dataset the EEG positions were digitized with a 3D pen: to get
# well-proportioned topographies we fit a sphere to the digitized points.

# stimiamo la sfera che meglio approssima la forma della testa, a partire dalle posizioni dei sensori EEG
radius, center, _ = mne.bem.fit_sphere_to_headshape(evokeds["famous"].info, dig_kinds="eeg")
sphere = tuple(center) + (radius,)
print(f"sphere radius: {radius * 1000:.1f} mm")

### 2.1 All three in a single plot

The **GFP** (*global field power*, the standard deviation across channels) summarizes
in a single curve how much activity there is in the electric field at each instant.

In [ ]:
# Qui confrontiamo le tre condizioni attraverso la Global Field Power.
# La GFP misura quanto è forte complessivamente la distribuzione del potenziale sulla testa
fig = mne.viz.plot_compare_evokeds(
    evokeds, picks="eeg", combine="gfp", colors=COLORS,
    title="EEG GFP - the three conditions", show_sensors=False, show=False,
)[0]
fig.savefig(FIG_DIR / "01_evoked_gfp.png", dpi=150, bbox_inches="tight")

In [ ]:
# Same comparison on a single posterior channel (where the N170 is clearly visible)

#Ora non usiamo più GFP. Confrontiamo direttamente i tre evoked sull'elettrodo EEG065
fig = mne.viz.plot_compare_evokeds(
    evokeds, picks="EEG065", colors=COLORS,
    title="Channel EEG065 - the three conditions", show=False,
)[0]
fig.savefig(FIG_DIR / "01_evoked_EEG065.png", dpi=150, bbox_inches="tight")

### 2.2 One condition at a time: butterfly + topographies

In [ ]:
#Per ogni condizione viene creato un plot_joint.
# Mostra contemporaneamente: risposta EEG nel tempo + topografia a 90 ms/170/250/400ms

for cond, ev in evokeds.items():
    ev.plot_joint(
        times=[0.09, 0.17, 0.25, 0.40],
        topomap_args=dict(sphere=sphere),
        title=f"EEG - {cond}",
    )

In [ ]:
# Topographies of the three conditions at the same instant (N170 peak)

#Troviamo automaticamente il picco N170 nella condizione famous e salviamo soltanto il tempo
_, peak_time = evokeds["famous"].get_peak(ch_type="eeg", tmin=0.12, tmax=0.22)
print(f"peak used for the topographies: {peak_time * 1000:.0f} ms")

fig, axes = plt.subplots(1, 4, figsize=(9, 3),
                         gridspec_kw={"width_ratios": [3, 3, 3, 0.4]})

#Topografie allo stesso istante
#average=0.03 media circa 30 ms intorno a quel punto.
for ii, cond in enumerate(CONDITIONS):
    evokeds[cond].plot_topomap(
        times=peak_time, average=0.03, sphere=sphere, vlim=(-4.5, 4.5),
        axes=(axes[2:] if ii == 2 else axes[ii]),
        colorbar=(ii == 2), show=False,
    )
    axes[ii].set_title(cond)
fig.suptitle(f"EEG, {peak_time * 1000:.0f} ms")
fig.savefig(FIG_DIR / "01_topomaps.png", dpi=150, bbox_inches="tight")

## 3. The inverse operator

The inverse operator combines three ingredients: the **forward solution** (the
physics), the **noise covariance** (the noise model) and the **prior assumptions**
about the sources.

Two parameters matter a lot:

* `loose` — how much the sources may deviate from the cortical normal
  (`0` = fixed orientation, `1` = fully free orientation);
* `depth` — depth-bias compensation, i.e. the tendency of minimum-norm methods to
  place activity too close to the surface.

For this hands-on, source orientations are **fixed to the cortical normal** (`loose = 0`) in order to match the constrained cortical source model used in the Brainstorm workflow.

We keep the depth-weighting parameter fixed: depth = 0.8

The inverse operator **does not depend on the evoked condition**, so we compute it once and apply it to all three conditions.



In [ ]:
# Fixed source orientation: dipoles are constrained to the cortical normal.

LOOSE = 0 # Controlla quanto le sorgenti possono deviare dalla normale corticale
DEPTH = 0.8 # Compensa il bias dei minimum-norm methods verso le sorgenti superficiali

inverse_operator = make_inverse_operator(
    evokeds["famous"].info, fwd, noise_cov, loose=LOOSE, depth=DEPTH,
)

inv_fname = RESULTS / f"{SUBJECT}-eeg-inv.fif"
write_inverse_operator(inv_fname, inverse_operator, overwrite=True)
print("inverse operator saved to", inv_fname.name)
inverse_operator

## 4. A first solution: dSPM with SNR = 3

Regularization is expressed through the expected signal-to-noise ratio:

$$\lambda^2 = \frac{1}{\mathrm{SNR}^2}$$

A **high** SNR means "I trust the data" → little regularization.
A **low** SNR means "I do not trust the data much" → strong regularization.

In [ ]:
# Time window over which we compute and save the source solutions.
# We keep a bit of baseline so we can check the background level.

#Calcolo di default del codice.
# Per ogni condizione:
#Evoked
#  ↓
#crop -0.1→0.5 s
#  ↓
#inverse operator
#  +
#lambda²
#  +
#dSPM
#  ↓
#SourceEstimate

STC_WINDOW = (-0.1, 0.5)

snr_ref = 3.0
lambda2_ref = 1.0 / snr_ref ** 2
method_ref = "dSPM"

stc_ref = {
    cond: apply_inverse(
        evokeds[cond].copy().crop(*STC_WINDOW), inverse_operator,
        lambda2_ref, method=method_ref
    )
    for cond in CONDITIONS
}

stc = stc_ref["famous"]
print(stc)
print(f"\ndata: {stc.data.shape[0]} sources x {stc.data.shape[1]} samples")

#Otteniamo: stc_ref["famous"], stc_ref["unfamiliar"], stc_ref["scrambled"]
#Ora siamo finalmente passati da: sensori x tempo a sorgenti corticali x tempo


In [ ]:
# The N170 source peak is defined as the maximum absolute cortical source activity within the time window [120, 200]ms

# Questa funzione cerca il massimo della ricostruzione tra 120−200ms.
# Otteniamo: idx   → indice spaziale del massimo, t_idx → indice temporale del massimo
# mode="abs" cerca il massimo in valore assoluto.

def peak_info(stc, tmin=0.12, tmax=0.2):
    """Return (hemisphere, vertex, latency, amplitude) of the absolute maximum."""
    idx, t_idx = stc.get_peak(tmin=tmin, tmax=tmax, mode="abs",
                              vert_as_index=True, time_as_index=True)
    n_lh = len(stc.vertices[0])
    if idx < n_lh:
        hemi, vertno = "lh", stc.vertices[0][idx]
    else:
        hemi, vertno = "rh", stc.vertices[1][idx - n_lh]
    return hemi, int(vertno), float(stc.times[t_idx]), float(stc.data[idx, t_idx])

for cond, s in stc_ref.items():
    hemi, vertno, lat, amp = peak_info(s)
    print(f"{cond:<12} peak in {hemi} vertex {vertno:>5} "
          f"at {lat * 1000:6.1f} ms (amplitude {amp:.2f})")



### 4.1 Visualization on the cortical surface

A separate interactive window opens: you can rotate the brain, move the time cursor,
play the activity as a movie. **Keep the window open** as long as you need it, then
close it.

In [ ]:
brain = abs(stc_ref["famous"]).plot(
    subject=SUBJECT,
    subjects_dir=SUBJECTS_DIR,
    surface="pial",
    hemi="lh",
    views="lat",
    initial_time=0.17,
    time_unit="s",
    time_viewer=False,
)


In [ ]:
brain = abs(stc_ref["famous"]).plot(
    subject=SUBJECT,
    subjects_dir=SUBJECTS_DIR,
    surface="inflated",
    hemi="lh",
    views="lat",
    initial_time=0.17,
    time_unit="s",
    time_viewer=False,
)

In [ ]:
#Visualizzazione della dSPM sulla corteccia
# Mostriamo la ricostruzione famous a: 170ms

brain = abs(stc_ref["famous"]).plot(
    subject=SUBJECT, subjects_dir=SUBJECTS_DIR,
    hemi="split", views=["lat", "med"],
    initial_time=0.17, time_unit="s",
    title="famous - dSPM, SNR=3",
)

## 5. Regions of interest (ROIs)

A cortical map is hard to compare by eye across methods. A more quantitative approach
is to extract the **mean time course inside anatomically defined regions**, using
FreeSurfer's `aparc` parcellation (Desikan-Killiany atlas).

In MNE a ROI is an `mne.Label` object; a collection of ROIs is an *annotation*.

Useful extraction modes:

* `mean` — plain average inside the ROI: careful, sources with opposite orientation
  cancel each other out;
* `mean_flip` — aligns the signs before averaging (usually the best choice);
* `pca_flip` — first principal component, with aligned sign;
* `max` — the maximum value inside the ROI.

In [ ]:
labels = mne.read_labels_from_annot(SUBJECT, parc="aparc", subjects_dir=SUBJECTS_DIR) #Carichiamo l'atlante FreeSurfer Desikan-Killiany
labels = [lab for lab in labels if "unknown" not in lab.name] #rimuoviamo le regioni unknown
print(f"{len(labels)} ROIs available in the aparc annotation")
print([lab.name for lab in labels[:8]], "...")

In [ ]:
# Visual/ventral ROIs, the ones expected in a face-recognition task

#Il notebook sceglie: sei ROI di interesse

ROI_NAMES = [
    "lateraloccipital-lh", "lateraloccipital-rh",
    "fusiform-lh", "fusiform-rh",
    "inferiortemporal-lh", "inferiortemporal-rh",
]
roi_labels = [lab for name in ROI_NAMES for lab in labels if lab.name == name] #estrae dall'intero atlante soltanto queste sei
assert len(roi_labels) == len(ROI_NAMES), "some ROI was not found"

src_inv = inverse_operator["src"]   # the source space used for the inversion
ROI_MODE = "mean_flip"


#Per ogni condizione otteniamo una matrice: 6ROI×Ntempi
roi_ref = {
    cond: stc_ref[cond].extract_label_time_course(roi_labels, src_inv, mode=ROI_MODE)
    for cond in CONDITIONS
}
print("shape of the extracted matrix (ROIs x times):", roi_ref["famous"].shape)

In [ ]:
#Rappresentazione che trasforma le complesse mappe corticali in sei semplici time course
times = stc_ref["famous"].times

fig, axes = plt.subplots(3, 2, figsize=(11, 8), sharex=True, sharey=True)
for ax, name, li in zip(axes.ravel(), ROI_NAMES, range(len(ROI_NAMES))):
    for cond in CONDITIONS:
        ax.plot(times, roi_ref[cond][li], label=cond, color=COLORS[cond], lw=1.6)
    ax.axhline(0, color="black", lw=0.8)
    ax.axvline(0, color="black", lw=0.8, ls=":")
    ax.set_title(name, fontsize=10)
axes[0, 0].legend(fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("time (s)")
for ax in axes[:, 0]:
    ax.set_ylabel(f"{method_ref} (a.u.)")
fig.suptitle(f"ROI time courses - {method_ref}, SNR={snr_ref:g}, mode='{ROI_MODE}'")
fig.tight_layout()
fig.savefig(FIG_DIR / "01_roi_reference.png", dpi=150, bbox_inches="tight")

## 6. Inverse method and regularization

This is the core of the exercise. We keep data, forward model and noise model fixed,
and change only two things:

* the **inverse method**: `MNE`, `dSPM`, `sLORETA`;
* the **regularization**: SNR ∈ {1, 3, 5}, i.e. λ² ∈ {1, 0.25, 0.111, 0.04}.

For each combination (3 methods × 3 SNRs × 3 conditions = 27 solutions) we save:

* the full source solution in `results/stc/`;
* the peak (hemisphere, vertex, latency, amplitude) in `results/summary_peaks.csv`;
* the ROI time courses in `results/roi_timecourses.csv`.

Each `.stc` file takes about 5 MB, so roughly 200 MB in total. If disk space is
tight, shorten `SNRS` or `CONDITIONS_GRID`.

In [ ]:
#CALCOLO GLI INVERSI
METHODS = ["MNE", "dSPM", "sLORETA"]
SNRS = [1.0, 3.0, 5.0]
CONDITIONS_GRID = CONDITIONS

#definiamo come salvare i file STC, in base a condizione, metodo, SNR
def stc_path(cond, method, snr):
    """Naming convention shared with notebook 02."""
    return STC_DIR / f"{SUBJECT}_cond-{cond}_method-{method}_snr-{snr:g}"

def spatial_extent(stc, time, threshold=0.5):
    t_idx = np.argmin(np.abs(stc.times - time))
    frame = np.abs(stc.data[:, t_idx])
    return float(np.mean(frame >= threshold * frame.max()))


peak_rows = []
roi_frames = []
extent_rows = []

for method in METHODS:
    for snr in SNRS:
        lambda2 = 1.0 / snr ** 2 
        for cond in CONDITIONS_GRID:
            stc = apply_inverse(
                evokeds[cond].copy().crop(*STC_WINDOW), inverse_operator,
                lambda2, method=method, pick_ori=None,
            )
            stc.save(stc_path(cond, method, snr), ftype="stc", overwrite=True)

            hemi, vertno, lat, amp = peak_info(stc)
            peak_rows.append(dict(
                condition=cond, method=method, snr=snr, lambda2=lambda2,
                peak_hemi=hemi, peak_vertex=vertno,
                peak_latency_s=lat, peak_amplitude=amp,
            ))

            # Spatial extent at 170 ms
            extent = spatial_extent(
                stc,
                time=0.170,
                threshold=0.5,
            )

            extent_rows.append(dict(
                condition=cond,
                method=method,
                snr=snr,
                time_s=0.170,
                threshold=0.5,
                extent_fraction=extent,
                extent_percent=100 * extent,
            ))

            tc = stc.extract_label_time_course(roi_labels, src_inv, mode=ROI_MODE)
            roi_frames.append(pd.DataFrame({
                "condition": cond, "method": method, "snr": snr,
                "roi": np.repeat(ROI_NAMES, tc.shape[1]),
                "time": np.tile(stc.times, tc.shape[0]),
                "value": tc.ravel(),
            }))
        print(f"  {method:<8} SNR={snr:g}  done")

print("\ngrid completed:", len(peak_rows), "solutions saved")

In [ ]:
#Registriamo i peak
peaks = pd.DataFrame(peak_rows)

# MNI coordinates of the peak: useful for a quantitative comparison across methods
def to_mni(row):
    try:
        coords = mne.vertex_to_mni(
            row["peak_vertex"], hemis=0 if row["peak_hemi"] == "lh" else 1,
            subject=SUBJECT, subjects_dir=SUBJECTS_DIR,
        )
        return pd.Series(np.round(np.atleast_2d(coords)[0], 1),
                         index=["mni_x", "mni_y", "mni_z"])
    except Exception:
        return pd.Series([np.nan] * 3, index=["mni_x", "mni_y", "mni_z"])

peaks = peaks.join(peaks.apply(to_mni, axis=1))
peaks.to_csv(RESULTS / "summary_peaks.csv", index=False)

extent_df = pd.DataFrame(extent_rows)
extent_df.to_csv(
    RESULTS / "summary_extent.csv",
    index=False,
)

roi_df = pd.concat(roi_frames, ignore_index=True)
roi_df.to_csv(RESULTS / "roi_timecourses.csv", index=False)

# Combine peak and extent information for inspection
summary = peaks.merge(
    extent_df[
        [
            "condition",
            "method",
            "snr",
            "extent_fraction",
            "extent_percent",
        ]
    ],
    on=["condition", "method", "snr"],
    how="left",
)

print(
    "saved: results/summary_peaks.csv, "
    "results/summary_extent.csv, "
    "results/roi_timecourses.csv"
)

summary.head(20)
#Per ciascuna soluzione conserviamo: condizione, metodo, SNR, lambda², emisfero del picco, vertice del picco, latenza, ampiezza

In [ ]:
# Quick look: how do peak latency and position change?
pd.set_option("display.width", 120)
print(
    peaks[peaks.condition == "famous"]
    .pivot(index="snr", columns="method", values="peak_latency_s")
    .round(3)
    .to_string()
)

## 7. Wrap-up and next notebook

At this point `results/` contains:

```
results/
├── sub-01-eeg-inv.fif       the inverse operator we used
├── stc/                     36 source solutions (.stc)
├── summary_peaks.csv        peak of each solution (+ MNI coordinates)
├── roi_timecourses.csv      ROI time courses for the whole grid
└── figures/                 sensor/ROI figures generated here
```

Now open **`02_visualization.ipynb`** to visually compare the cortical maps and the
time courses across methods and regularization levels.

### Before you close
Write down somewhere (lab notebook, a text cell, the README): the method and SNR you
used, `loose` and `depth`, the noise covariance window, the ROI extraction mode.
These are exactly the parameters that make — or fail to make — a source
reconstruction result reproducible.

---

# 8. Your turn

So far you have run somebody else's choices. From here on **you** decide what to
reconstruct and how. Two ways of doing it:

* **8.1-8.3**: fill in the blanks (`...`) and run the cells. This is the version
  you should be able to write from scratch by the end of the session.
* **8.4**: the same thing with menus and sliders, to explore many combinations
  quickly.

Nothing you do below overwrites the grid saved in section 6: your solutions are
kept in separate variables and, if you want, saved with a `my-` prefix.

## 8.1 Make your choices

Replace each `...` with a value. You only choose **three** things: condition, method
and SNR. The regularization parameter is not a free choice — it follows from the SNR
through λ² = 1/SNR², so the cell computes it for you.

The helper tells you whether the choices make sense before you spend time computing
anything.

In [ ]:
#controlla che:
#la condizione esista;
#il metodo sia valido;
#SNR sia positivo.

def check_choices(condition=None, method=None, snr=None):
    """Small sanity check on the student's choices."""
    known_methods = ["MNE", "dSPM", "sLORETA"]
    problems = []

    if condition is Ellipsis or condition is None:
        problems.append("MY_CONDITION is still empty")
    elif condition not in CONDITIONS:
        problems.append(f"MY_CONDITION={condition!r}: pick one of {CONDITIONS}")

    if method is Ellipsis or method is None:
        problems.append("MY_METHOD is still empty")
    elif method not in known_methods:
        problems.append(f"MY_METHOD={method!r}: pick one of {known_methods}")

    if snr is Ellipsis or snr is None:
        problems.append("MY_SNR is still empty")
    elif not isinstance(snr, (int, float)) or snr <= 0:
        problems.append(f"MY_SNR={snr!r}: it must be a positive number")

    if problems:
        print("Not there yet:")
        for p in problems:
            print("  -", p)
        return False

    print(f"All good: {condition}, {method}, SNR={snr:g}")
    return True

In [ ]:
# ----------------------------------------------------------------------
# TODO 1 - which evoked response do you want to reconstruct?
#          one of "famous", "unfamiliar", "scrambled"
MY_CONDITION = ...

# TODO 2 - which inverse method? "MNE", "dSPM", "sLORETA" (try "eLORETA" too)
MY_METHOD = ...

# TODO 3 - how much do you trust the data? any positive number, e.g. 1, 3, 5
MY_SNR = ...

# Regularization: derived from the SNR, nothing to fill in here.
if check_choices(MY_CONDITION, MY_METHOD, MY_SNR):
    MY_LAMBDA2 = 1.0 / MY_SNR ** 2
    print(f"  lambda2 = 1 / {MY_SNR:g}^2 = {MY_LAMBDA2:.4f}")

## 8.2 Compute your solution

Two blanks left: the evoked response to feed in, and the method to use.

In [ ]:
stc_mine = apply_inverse(
    evokeds[MY_CONDITION].copy().crop(*STC_WINDOW),   
    inverse_operator,
    MY_LAMBDA2,
    method= MY_METHOD,                              
)

hemi, vertno, lat, amp = peak_info(stc_mine)
print(f"{MY_METHOD}, SNR={MY_SNR:g}, condition '{MY_CONDITION}'")
print(f"  peak in {hemi} vertex {vertno} at {lat * 1000:.1f} ms (amplitude {amp:.2f})")

# Optional: save it next to the others, with a 'my-' prefix so it is easy to spot
# stc_mine.save(STC_DIR / f"my-{MY_CONDITION}_{MY_METHOD}_snr-{MY_SNR:g}",
#               ftype="stc", overwrite=True)

In [ ]:
# ROI time courses of YOUR solution, next to the reference one (dSPM, SNR=3)

#Il riferimento è sempre: dSPM, SNR=3
#La propria soluzione viene normalizzata e sovrapposta alla reference.
#Questo permette di rispondere visivamente:
#scegliendo MNE/SLORETA oppure cambiando SNR, quanto cambia la forma temporale nelle stesse ROI?

tc_mine = stc_mine.extract_label_time_course(roi_labels, src_inv, mode=ROI_MODE)

fig, axes = plt.subplots(3, 2, figsize=(11, 8), sharex=True)
for ax, name, li in zip(axes.ravel(), ROI_NAMES, range(len(ROI_NAMES))):
    ref = roi_ref[MY_CONDITION][li]
    ax.plot(times, ref / np.abs(ref).max(), color="grey", lw=1.2,
            label=f"{method_ref}, SNR={snr_ref:g}")
    ax.plot(times, tc_mine[li] / np.abs(tc_mine[li]).max(),
            color="crimson", lw=1.8, label=f"{MY_METHOD}, SNR={MY_SNR:g}")
    ax.axhline(0, color="black", lw=0.8)
    ax.axvline(0, color="black", lw=0.8, ls=":")
    ax.set_title(name, fontsize=10)
axes[0, 0].legend(fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("time (s)")
for ax in axes[:, 0]:
    ax.set_ylabel("normalized amplitude")
fig.suptitle(f"Your solution vs the reference one - condition '{MY_CONDITION}'")
fig.tight_layout()

## 8.3 Look at it on the cortex

One more blank: the latency you want to look at. Pick it from the peak printed
above, or from the ROI plot.

In [ ]:
MY_TIME = ...   # TODO 4 - latency in seconds, e.g. 0.17

brain_mine = abs(stc_mine).plot(
    subject=SUBJECT, subjects_dir=SUBJECTS_DIR,
    hemi="both", views="lateral",
    initial_time=MY_TIME, time_unit="s",
    time_viewer=True, show_traces=True,
    size=(900, 650), background="white",
    title=f"{MY_CONDITION} - {MY_METHOD}, SNR={MY_SNR:g}",
)

In [ ]:
# brain_mine.close()   # uncomment to close the window

## 8.4 The same thing with menus and sliders

If you would rather try ten combinations in two minutes, use the controls below.
Set condition, method, SNR and ROI, then press **Run Interact**: the inverse solution
is recomputed on the spot (it takes a second or two).

Note that the SNR slider is logarithmic: the interesting range spans two orders of
magnitude, and going from 1 to 2 matters far more than going from 8 to 9.

In [ ]:
#Stesso calcolo, approccio diverso
import ipywidgets as widgets

w_condition = widgets.Dropdown(options=CONDITIONS, value="famous", description="condition:")
w_method = widgets.Dropdown(options=["MNE", "dSPM", "sLORETA", "eLORETA"],
                            value="dSPM", description="method:")
w_snr = widgets.FloatLogSlider(value=3.0, base=10, min=-0.5, max=1.2, step=0.02,
                               description="SNR:", readout_format=".2f",
                               continuous_update=False)
w_roi = widgets.Dropdown(options=ROI_NAMES, value="fusiform-rh", description="ROI:")


def explore_inverse(condition, method, snr, roi):
    """Recompute one solution and show the ROI time course + the peak."""
    lambda2 = 1.0 / snr ** 2
    stc = apply_inverse(evokeds[condition].copy().crop(*STC_WINDOW),
                        inverse_operator, lambda2, method=method)
    tc = stc.extract_label_time_course(roi_labels, src_inv, mode=ROI_MODE)
    li = ROI_NAMES.index(roi)
    hemi, vertno, lat, amp = peak_info(stc)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(stc.times, tc[li], lw=1.8, color="crimson")
    ax.axhline(0, color="black", lw=0.8)
    ax.axvline(0, color="black", lw=0.8, ls=":")
    ax.axvline(lat, color="steelblue", lw=1.2, ls="--",
               label=f"whole-brain peak: {lat * 1000:.0f} ms ({hemi})")
    ax.set_xlabel("time (s)")
    ax.set_ylabel(f"{method} (a.u.)")
    ax.set_title(f"{condition} - {method}, SNR={snr:.2f} (lambda2={lambda2:.3f}) - {roi}")
    ax.legend(fontsize=8)
    plt.show()

    global stc_widget
    stc_widget = stc    # kept aside, in case you want to plot it in 3D


widgets.interact_manual(explore_inverse, condition=w_condition, method=w_method,
                        snr=w_snr, roi=w_roi);

In [ ]:
# The last solution computed with the widgets stays in `stc_widget`:
# uncomment to open it in the 3D viewer.
stc_widget.plot(subject=SUBJECT, subjects_dir=SUBJECTS_DIR, hemi="both",
                 initial_time=0.17, time_viewer=True, show_traces=True)